# Fase 3 — Validación y Comparativa de modelos

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook:
1. Loads all saved predictions and metrics (Naive t-168, SARIMAX, LSTM, TTM zero-shot, TTM few-shot)
2. Computes a unified metrics table (MAPE, RMSE, MAE, training time, inference time)
3. Produces publication-quality comparison plots (Weekly actual vs predicted, Bar charts, Scatter plots, Computational efficiency)
4. Exports final comparison tables to CSV and Google Drive

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
sns.set_theme(style='whitegrid')
print('Environment ready.')

## 1 · Load saved predictions & metrics

In [ ]:
def load_preds(path):
    df = pd.read_csv(path)
    # Standardize to tz-naive UTC datetime to prevent tz-naive vs tz-aware comparison errors
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_localize(None)
    df = df.sort_values('datetime').reset_index(drop=True)
    return df

pred_files = {
    'Naive (t-168)' : 'data/predictions_naive.csv',
    'SARIMAX'       : 'data/predictions_sarimax.csv',
    'LSTM'          : 'data/predictions_lstm.csv',
    'TTM zero-shot' : 'data/predictions_ttm_zeroshot.csv',
    'TTM few-shot'  : 'data/predictions_ttm_fewshot.csv',
}

preds = {}
for name, path in pred_files.items():
    if os.path.exists(path):
        preds[name] = load_preds(path)
        print(f'{name:<18}: {len(preds[name]):,} rows  ({preds[name]["datetime"].min().date()} → {preds[name]["datetime"].max().date()})')
    else:
        print(f'{name:<18}: NOT FOUND — run the corresponding notebook first')

In [ ]:
# Load metrics JSON files
metric_files = {
    'Naive (t-168)' : 'data/metrics_naive.json',
    'SARIMAX'       : 'data/metrics_sarimax.json',
    'LSTM'          : 'data/metrics_lstm.json',
    'TTM zero-shot' : 'data/metrics_ttm_zeroshot.json',
    'TTM few-shot'  : 'data/metrics_ttm_fewshot.json',
}

all_metrics = {}
for name, path in metric_files.items():
    if os.path.exists(path):
        with open(path) as f:
            all_metrics[name] = json.load(f)
        print(f'{name:<18}: MAPE={all_metrics[name].get("mape", 0):.3f}% | RMSE={all_metrics[name].get("rmse", 0):.1f} MW | Train={all_metrics[name].get("train_s", 0):.1f}s | Infer={all_metrics[name].get("inference_s", 0):.3f}s')
    else:
        print(f'{name:<18}: metrics file NOT FOUND')

# Fallback: if predictions exist but metrics file was not saved, compute directly
for name, df in preds.items():
    if name not in all_metrics:
        m = du.compute_metrics(df['y_true'].values, df['y_pred'].values, label=name)
        m['train_s'] = 0.0
        m['inference_s'] = 0.0
        all_metrics[name] = m

## 2 · Unified Metrics Comparison Table

In [ ]:
summary_df = du.plot_comparison_table(
    all_metrics,
    figsize = (12, max(3, len(all_metrics) * 0.6 + 1)),
    save_path = 'data/fig_comparison_table.png',
)
summary_df.to_csv('data/comparison_table.csv')
print('\nSaved → data/comparison_table.csv')

## 3 · Visual Comparison

In [ ]:
# Choose a representative week for plotting from the first available model
if preds:
    first_model = list(preds.keys())[0]
    df_ref = preds[first_model]
    dates_ref = df_ref['datetime']

    PLOT_DAYS  = 7
    PLOT_START = dates_ref.min() + pd.Timedelta(days=30)
    PLOT_END   = PLOT_START + pd.Timedelta(days=PLOT_DAYS)

    model_colors = {
        'Naive (t-168)': '#7f7f7f',
        'SARIMAX'      : '#e377c2',
        'LSTM'         : '#d62728',
        'TTM zero-shot': '#2ca02c',
        'TTM few-shot' : '#17becf',
    }

    fig, ax = plt.subplots(figsize=(15, 5))

    # Plot actual demand from reference model
    mask_ref = (dates_ref >= PLOT_START) & (dates_ref < PLOT_END)
    ax.plot(dates_ref[mask_ref], df_ref['y_true'][mask_ref], label='Actual', color='#1f77b4', linewidth=2.0, zorder=10)

    # Overlay model predictions
    for name, df in preds.items():
        d = df['datetime']
        mask = (d >= PLOT_START) & (d < PLOT_END)
        if mask.sum() > 0:
            ax.plot(d[mask], df['y_pred'][mask], label=name,
                    color=model_colors.get(name, 'gray'), linewidth=1.3, linestyle='--', alpha=0.9)

    ax.set_title(f'All models — actual vs predicted ({PLOT_START.date()} to {PLOT_END.date()})')
    ax.set_ylabel('Demand (MW)')
    ax.legend(loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b %Hh'))
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.savefig('data/fig_all_models_week.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Bar charts: MAPE, RMSE, MAE across models
if all_metrics:
    names  = list(all_metrics.keys())
    mapes  = [all_metrics[n]['mape'] for n in names]
    rmses  = [all_metrics[n]['rmse'] for n in names]
    maes   = [all_metrics[n]['mae']  for n in names]

    x = np.arange(len(names))
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    for ax, vals, ylabel, title in [
        (axes[0], mapes, 'MAPE (%)',  'MAPE by model (lower is better)'),
        (axes[1], rmses, 'RMSE (MW)', 'RMSE by model (lower is better)'),
        (axes[2], maes,  'MAE (MW)',  'MAE by model (lower is better)'),
    ]:
        bars = ax.bar(x, vals, color=[model_colors.get(n, '#7f7f7f') for n in names], alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=20, ha='right')
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig('data/fig_metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Scatter plot: actual vs predicted for all available models
if preds:
    N_SCATTER = 2000
    n_cols    = len(preds)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5), sharey=True)

    if n_cols == 1:
        axes = [axes]

    for ax, (name, df) in zip(axes, preds.items()):
        idx_sample = np.random.choice(len(df), size=min(N_SCATTER, len(df)), replace=False)
        yt = df['y_true'].values[idx_sample]
        yp = df['y_pred'].values[idx_sample]
        ax.scatter(yt, yp, alpha=0.2, s=5, color=model_colors.get(name, 'gray'))
        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        ax.plot(lims, lims, 'k--', linewidth=1)
        ax.set_title(name)
        ax.set_xlabel('Actual (MW)')
        m = all_metrics.get(name, {})
        ax.text(0.05, 0.95, f"MAPE={m.get('mape', 0):.2f}%\nRMSE={m.get('rmse', 0):.0f} MW",
                transform=ax.transAxes, va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    axes[0].set_ylabel('Predicted (MW)')
    plt.suptitle('Actual vs Predicted — all models', y=1.01)
    plt.tight_layout()
    plt.savefig('data/fig_scatter_all_models.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4 · Computational Efficiency Comparison

In [ ]:
# Computational efficiency: Training time & Inference time side-by-side
if all_metrics:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    # 1. Training time (minutes)
    train_times = {n: all_metrics[n].get('train_s', 0) for n in all_metrics}
    axes[0].barh(
        list(train_times.keys()),
        [v / 60 for v in train_times.values()],
        color=[model_colors.get(n, 'gray') for n in train_times],
        alpha=0.85,
    )
    axes[0].set_xlabel('Training time (minutes)')
    axes[0].set_title('Training Time per Model (minutes)')
    for i, (k, v) in enumerate(train_times.items()):
        axes[0].text(v / 60 + 0.05, i, f'{v/60:.2f} min ({v:.1f}s)', va='center', fontsize=9)

    # 2. Inference time (seconds)
    infer_times = {n: all_metrics[n].get('inference_s', 0) for n in all_metrics}
    axes[1].barh(
        list(infer_times.keys()),
        list(infer_times.values()),
        color=[model_colors.get(n, 'gray') for n in infer_times],
        alpha=0.85,
    )
    axes[1].set_xlabel('Inference time (seconds)')
    axes[1].set_title('Inference Time per Model (seconds)')
    for i, (k, v) in enumerate(infer_times.items()):
        axes[1].text(v + 0.01, i, f'{v:.3f}s', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig('data/fig_computational_efficiency.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5 · Final Results Table & Export

In [ ]:
rows = []
for name, m in all_metrics.items():
    rows.append({
        'Model'          : name,
        'MAPE (%)'       : round(m['mape'],  3),
        'RMSE (MW)'      : round(m['rmse'],  1),
        'MAE (MW)'       : round(m['mae'],   1),
        'Train time (s)' : round(m.get('train_s', 0), 1),
        'Inference (s)'  : round(m.get('inference_s', 0), 3),
    })

final_table = pd.DataFrame(rows).set_index('Model')
print(final_table.to_string())
final_table.to_csv('data/final_results_table.csv')
print('\nSaved → data/final_results_table.csv')

# Sync outputs to Google Drive (if in Colab)
du.sync_to_drive()


## Summary

| Model | MAPE (%) | RMSE (MW) | MAE (MW) | Train (min) | Inference (s) |
|-------|:--------:|:---------:|:--------:|:-----------:|:-------------:|
| **Naive (t-168)** | … | … | … | **0.0** | **< 0.05** |
| **SARIMAX**        | … | … | … | …       | …          |
| **LSTM**           | … | … | … | …       | …          |
| **TTM zero-shot**  | … | … | … | **0.0** | …          |
| **TTM few-shot**   | … | … | … | …       | …          |

**Best model:** …  
**Conclusions:** …

In [ ]:
#Terminate runtime
from google.colab import runtime
runtime.unassign()